# 11주차 랩 · 시계열 예측

**⏱ 60분** · 미국 월별 철도 여객마일 1990년 1월 ~ 2004년 4월 · 개인 제출

## 🎯 이번 랩의 질문

교재에서는 2001년 8월까지의 기록으로 항공 여객의 기준선을 예측해, 2001년 9월 이후 첫 1년 동안 15.6% 적었다는 것을 확인했습니다. 항공 대신 철도를 이용한 사람이 늘었을 수도 있고, 철도 이용도 함께 줄었을 수도 있습니다. 같은 자료의 철도 여객마일로 같은 절차를 밟습니다.

> **2001년 9월 이후 미국 철도 여객은 이전 추세로 예측한 기준선과 달라졌을까요?**

| 패턴 | 교재 | 이번 랩 |
|---|---|---|
| 2001년 8월까지·9월 이후, 학습·테스트 기간 나누기 | 2절 | 과제 1 |
| naive 예측, seasonal naive 예측 | 3절 | 과제 2 |
| lag 열, 자기회귀 AR(4)의 다단계 예측, 계절 자기회귀 | 4절 | 과제 3 |
| 회귀용 표, 선형 추세 모형, 선형 추세 + 계절성 모형 | 5절 | 과제 4 |
| 오차 지표로 비교, 학습 오차와 테스트 오차 | 6절 | 과제 5 |
| 기준선과 실제의 차이 | 7·8절 | 과제 6 |
| 숫자를 근거로 결정 | 8절 | 과제 7 |

### 진행 방법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행합니다 |
| 📝 | 과제 | `None` 자리를 코드로 바꿔 채웁니다. 변수 이름은 바꾸지 않습니다 |
| ✅ | 확인 셀 | 고치지 말고 실행만 합니다 |
| 💬 | 결과 읽기 | 결과에서 확인할 숫자를 짚습니다 |

- 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. 결과는 반올림하지 않고 변수에 저장합니다.
- 확인 셀은 변수의 형식만 봅니다. "형식 확인 완료"는 값이 맞다는 뜻이 아닙니다. 값은 제출 후 채점합니다.

## ▶ 준비

교재와 같은 BTS 월별 교통량에서 철도 여객마일(`Rail PM`, 승객 1명을 1마일 태우면 1)만 씁니다. 100만으로 나눠 **백만 여객마일** 단위로 둡니다.

준비 셀이 만드는 것:
- `df` — 한 행이 한 달인 표. 날짜가 index이고 열은 `rail` 하나입니다(교재 1절에서 만든 모양과 같음).
- `errors(y_true, y_pred)` — 교재 6절의 오차 함수와 같습니다. MSE·MAPE를 돌려줍니다.
- `preds`, `fits` — 방법 이름을 붙여 테스트 기간의 예측값과 학습 기간의 적합값을 모을 곳입니다(교재 3절과 같음).
- `plot_forecast(name)` — 교재 3절의 그림 함수와 같습니다. 실제 값(회색 점), 적합값(실선), 테스트 기간의 예측(점선)을 그립니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 철도 여객마일을 읽습니다.
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import load, checker, summary, check_format
check = checker("week11")

travel = load("sept11_travel")
travel["Month"] = pd.to_datetime(travel["Month"], format="%b-%y")
df = travel.set_index("Month")[["Rail PM"]] / 1e6
df.columns = ["rail"]


def errors(y_true, y_pred):
    e = y_true - y_pred
    return pd.Series({"MSE": (e ** 2).mean(),
                      "MAPE": (e / y_true).abs().mean() * 100})


preds = {}
fits = {}


def plot_forecast(name):
    if preds.get(name) is None:
        print(name, ": 과제를 먼저 작성하세요.")
        return
    plt.figure(figsize=(9, 3.5))
    plt.plot(pre.loc["1997":].index, pre.loc["1997":, "rail"], "o", color="gray", markersize=3, label="Actual")
    if fits.get(name) is not None:
        plt.plot(fits[name].loc["1997":].index, fits[name].loc["1997":], color="C0", label="Fitted (training)")
    plt.plot(preds[name].index, preds[name], "--o", color="C0", markersize=4, label="Forecast (test)")
    plt.axvline(test.index[0], color="black", linewidth=0.8)
    plt.title(name)
    plt.ylabel("Rail passenger miles (millions)")
    plt.legend()
    plt.show()


plt.figure(figsize=(9, 3.5))
plt.plot(df.index, df["rail"])
plt.axvline(pd.Timestamp("2001-09-01"), color="red", linestyle="--")
plt.ylabel("Rail passenger miles (millions)")
plt.show()
print("준비 완료!", df.shape)

✅ `준비 완료! (172, 1)`이 나오면 됩니다.

💬 철도 여객은 1990년대 중반까지 줄었다가 다시 늘어납니다. 해마다 같은 모양의 오르내림도 있습니다. 2001년 9월의 움직임이 항공처럼 크게 보이나요?

---
## 📝 과제 1 · 2001년 8월까지·9월 이후와 학습·테스트 기간 나누기

기준선은 2001년 8월까지의 기록으로만 만들고, 방법은 그 마지막 12개월을 얼마나 잘 맞히는지로 고릅니다. 교재 2절처럼 나눕니다.
- 2001년 8월까지를 `pre`, 2001년 9월부터를 `post`로 나눕니다.
- `pre`의 마지막 12개월을 테스트 기간 `test`, 그 앞을 학습 기간 `train`으로 두고, 테스트 기간의 실제 값을 `y_test`에 담습니다.
- `pre`의 개월 수를 `n_pre`, `train`의 개월 수를 `n_train`에 담습니다.

**결과 형식:** 정수 두 개

In [ ]:
# 📝 과제 1 — None 자리를 채우세요
# 힌트: 교재 2절의 두 줄(날짜로 자르기), 그다음 세 줄(위치로 자르기, 테스트 기간의 실제 값)과 같습니다. 개월 수는 행의 수입니다.
pre = None
post = None
train = None
test = None
y_test = None
n_pre = None
n_train = None

In [ ]:
# ✅ 확인 — 과제 1 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(1, n_pre=n_pre, n_train=n_train)
else:
    check_format(1, n_pre=n_pre, n_train=n_train)

💬 **결과 읽기** — 테스트 기간은 언제부터 언제까지인가요?

---
## 📝 과제 2 · naive 예측과 seasonal naive 예측

교재 3절처럼 두 기준 예측을 만들어 `preds`에 담고, 준비 셀의 `errors`로 테스트 기간 MAPE를 구합니다.
- naive 예측을 `preds["naive"]`, seasonal naive 예측을 `preds["seasonal naive"]`에 담습니다(테스트 기간 index의 Series).
- 두 예측의 테스트 MAPE를 `naive_mape`, `snaive_mape`에 담습니다.

**결과 형식:** 숫자 두 개

In [ ]:
# 📝 과제 2 — None 자리를 채우세요
# 힌트: 교재 3절의 두 셀에서 열 이름 air를 rail로 바꿉니다. MAPE는 준비 셀의 errors에 테스트 기간의 실제 값과 예측을 넣어 나온 결과의 MAPE 칸입니다.
preds["naive"] = None
preds["seasonal naive"] = None
naive_mape = None
snaive_mape = None

In [ ]:
# ✅ 확인 — 과제 2 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(2, naive_mape=(naive_mape, 0.01), snaive_mape=(snaive_mape, 0.01))
else:
    check_format(2, naive_mape=naive_mape, snaive_mape=snaive_mape)

In [ ]:
# ▶ 그대로 실행하세요 — 두 기준 예측의 그림
plot_forecast("naive")
plot_forecast("seasonal naive")

💬 **결과 읽기** — 두 기준 예측은 실제 값의 몇 % 틀렸나요? 교재의 항공(14.22%, 2.74%)과 비교해 보세요.

---
## 📝 과제 3 · 자기회귀 AR(4)와 계절 자기회귀

교재 4절처럼 전체 표 `df`에 직전 값 열(`lag1`~`lag4`, `lag12`)을 붙이고, 두 자기회귀를 맞춰 테스트 기간을 예측합니다.
- AR(4)는 직전 4개월을 입력으로 맞추고, 테스트 기간은 앞 달의 예측값을 다음 달의 입력으로 이어 쓰는 다단계 예측으로 냅니다. 적합값을 `fits["AR(4)"]`, 예측을 `preds["AR(4)"]`에 담습니다.
- 계절 자기회귀는 `lag12` 하나를 입력으로 맞춥니다. 적합값을 `fits["seasonal AR"]`, 예측을 `preds["seasonal AR"]`에 담습니다.
- 두 예측의 테스트 MAPE를 `ar4_mape`, `seasonal_ar_mape`에 담습니다.

**결과 형식:** 숫자 두 개

In [ ]:
# 📝 과제 3 — None 자리를 채우세요
# 힌트: 교재 4절의 셀을 차례로 쓰고 열 이름 air를 rail로 바꿉니다. 학습 기간의 행은 df.loc[train.index]로 꺼내고, 입력이 모자란 처음 몇 달은 뺍니다. AR(4)의 테스트 기간 예측은 교재의 history 반복 셀과 같습니다.
fits["AR(4)"] = None
preds["AR(4)"] = None
fits["seasonal AR"] = None
preds["seasonal AR"] = None
ar4_mape = None
seasonal_ar_mape = None

In [ ]:
# ✅ 확인 — 과제 3 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(3, ar4_mape=(ar4_mape, 0.01), seasonal_ar_mape=(seasonal_ar_mape, 0.01))
else:
    check_format(3, ar4_mape=ar4_mape, seasonal_ar_mape=seasonal_ar_mape)

In [ ]:
# ▶ 그대로 실행하세요 — 두 자기회귀의 그림
plot_forecast("AR(4)")
plot_forecast("seasonal AR")

💬 **결과 읽기** — 두 자기회귀의 테스트 MAPE는 과제 2의 seasonal naive 예측보다 큰가요, 작은가요? 그림에서 AR(4)의 예측은 계절 모양을 그리나요?

---
## 📝 과제 4 · 추세·계절성 회귀

교재 5절처럼 전체 표 `df`에 `t`와 계절성 dummy 변수 11개(1월 기준)를 붙이고, 두 모형을 학습 기간 전체로 맞춰 테스트 기간을 예측합니다.
- 선형 추세 모형(`t`), 선형 추세 + 계절성 모형(`cols`: `t`와 계절성 dummy 변수)의 적합값과 예측을 `fits`, `preds`에 `"linear"`, `"linear + season"`이라는 이름으로 담습니다.
- 두 예측의 테스트 MAPE를 `linear_mape`, `linear_season_mape`에 담습니다.

**결과 형식:** 숫자 두 개

In [ ]:
# 📝 과제 4 — None 자리를 채우세요
# 힌트: 교재 5절의 회귀용 표 셀과 두 모형의 셀을 차례로 쓰고 열 이름 air를 rail로 바꿉니다. 맞힐 값은 학습 기간의 rail 열입니다.
cols = None
preds["linear"] = None
preds["linear + season"] = None
linear_mape = None
linear_season_mape = None

In [ ]:
# ✅ 확인 — 과제 4 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(4, linear_mape=(linear_mape, 0.01), linear_season_mape=(linear_season_mape, 0.01))
else:
    check_format(4, linear_mape=linear_mape, linear_season_mape=linear_season_mape)

In [ ]:
# ▶ 그대로 실행하세요 — 두 회귀 모형의 그림
plot_forecast("linear")
plot_forecast("linear + season")

💬 **결과 읽기** — 두 모형 중 테스트 MAPE가 더 작은 것은 무엇인가요?

---
## 📝 과제 5 · 오차 지표로 비교하기

교재 6절처럼 `preds`에 모은 모든 방법의 테스트 기간 오차를 한 표로 모으고, 적합값이 있는 방법은 학습 기간 MAPE와 나란히 봅니다.
- 모든 방법의 `errors` 결과를 행으로 모은 표를 `table`, 학습·테스트 MAPE 표(열 `"train"`, `"test"`)를 `mape`에 담습니다.
- 테스트 MAPE가 가장 작은 방법의 이름을 `best`, 학습 MAPE가 가장 작은 방법의 이름을 `best_train`에 담습니다. `idxmin()`이 가장 작은 값의 index를 돌려줍니다.

**결과 형식:** `table`, `mape`는 표, `best`, `best_train`은 문자열

In [ ]:
# 📝 과제 5 — None 자리를 채우세요
# 힌트: 교재 6절의 두 표 셀에서 air를 rail로 바꿉니다. 가장 작은 방법은 표의 MAPE 열(또는 train 열)에 idxmin()을 붙입니다.
table = None
mape = None
best = None
best_train = None

In [ ]:
# ✅ 확인 — 과제 5 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(5, best=best, best_train=best_train)
else:
    check_format(5, best=best, best_train=best_train)

💬 **결과 읽기** — 철도에서 기준선을 만들 방법은 무엇인가요? 학습 MAPE가 가장 작은 방법과 같은가요? 교재의 항공과 같은 방법인가요?

---
## 📝 과제 6 · 기준선과 실제의 차이

고른 방법의 기준선으로 2001년 9월 이후의 차이를 잽니다. 과제 5에서 고른 방법으로 첫 1년(2001년 9월 ~ 2002년 8월)의 기준선을 만들고 실제와 비교합니다. seasonal naive 예측이라면 2001년 8월까지의 마지막 12개월(2000년 9월 ~ 2001년 8월)의 값을 그대로 씁니다(교재 8절 마지막 셀).
- 첫 1년의 기준선을 `snaive_base`에 담습니다(첫 1년 index의 Series).
- 2001년 9월의 차이(실제 − 기준선)를 기준선 대비 %로 `sep_pct`에 담습니다.
- 첫 1년 차이의 합을 기준선 합계 대비 %로 `loss_pct`에 담습니다.

**결과 형식:** 숫자 두 개

In [ ]:
# 📝 과제 6 — None 자리를 채우세요
# 힌트: 교재 8절의 first_year 셀과 마지막 셀을 rail로 바꿔 씁니다. 2001년 9월은 첫 1년의 첫 달입니다.
first_year = None
snaive_base = None
sep_pct = None
loss_pct = None

In [ ]:
# ✅ 확인 — 과제 6 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(6, sep_pct=(sep_pct, 0.05), loss_pct=(loss_pct, 0.05))
else:
    check_format(6, sep_pct=sep_pct, loss_pct=loss_pct)

과제 6까지 채웠으면, 테스트 기간에서 고르지 않은 선형 추세 + 계절성 모형으로 기준선을 만들면 차이가 어떻게 나오는지 함께 봅니다. 교재 7절처럼 2001년 8월까지 전체로 다시 맞춘 기준선입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 선형 추세 + 계절성 모형의 기준선과 첫 1년의 비교
if snaive_base is not None:
    model = LinearRegression().fit(df.loc[pre.index, cols], pre["rail"])
    baseline = pd.Series(model.predict(df.loc[post.index, cols]), index=post.index)
    loss_pct_linear = (post["rail"] - baseline).loc[first_year].sum() / baseline.loc[first_year].sum() * 100
    print("선형 추세 + 계절성 모형 기준선으로 본 첫 1년의 차이(%):", round(loss_pct_linear, 2))
    plt.figure(figsize=(9, 3.5))
    plt.plot(df.loc["1999":"2002-08"].index, df.loc["1999":"2002-08", "rail"], color="gray", label="Actual")
    plt.plot(snaive_base.index, snaive_base, linestyle="--", label="Baseline: seasonal naive")
    plt.plot(baseline.loc[first_year].index, baseline.loc[first_year], linestyle=":", label="Baseline: linear + season")
    plt.axvline(pd.Timestamp("2001-09-01"), color="red", linestyle="--")
    plt.ylabel("Rail passenger miles (millions)")
    plt.legend()
    plt.show()
else:
    print("과제 6을 먼저 작성하세요.")

💬 **결과 읽기** — 고른 기준선으로 본 2001년 9월과 첫 1년의 차이는 몇 %인가요? 선형 추세 + 계절성 모형의 기준선으로 보면 몇 %인가요?

---
## 📝 과제 7 · 본인의 판단

1. 2001년 9월 이후 철도 여객은 기준선과 비교해 달라졌다고 보나요? 과제 6의 숫자 하나로 근거를 적으세요.
2. 선형 추세 + 계절성 모형의 기준선으로 보면 결론이 어떻게 달라지나요? 그 기준선을 쓰지 않는 이유를 과제 5의 숫자로 적으세요.

자동 채점은 하지 않습니다.

✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

---
## ✅ 제출 전 확인

아래 셀을 실행해 과제 1~6의 작성 여부와 형식을 확인합니다. "형식 확인 완료"는 계산값이 맞다는 뜻이 아닙니다. 결과를 직접 읽고 과제 7을 작성해 제출합니다. 실제 값 채점은 제출 후 진행합니다.

In [ ]:
# ✅ 제출 전 확인 — 과제 1~6의 작성 여부와 형식
summary()